# SD_5min Edge Delay Analysis

这个 notebook 直接在 `SD_5min` 上，按论文中的 `max-cross-correlation (MCC)` 口径分析边传播 delay 分布。

这里采用更贴近论文的实现：先用 **natural cubic spline** 生成连续路径，再在更细时间网格上搜索 delay。


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
sys.path.append(str(REPO_ROOT / 'benchmark' / 'eval'))

from old_sd_analysis_utils import (
    DEFAULT_SD_5MIN_DIR,
    DEFAULT_SD_5MIN_GRAPH_ROOT,
    build_sd_5min_graph_matrices,
    compute_daily_delay_distributions,
    compute_delay_distribution,
    load_dataset_flow,
    load_sensor_ids,
    slice_flow_by_day_window,
    summarize_delay_distribution,
)

sns.set_theme(style='whitegrid')


In [ ]:
# Config
SD_5MIN_DIR = DEFAULT_SD_5MIN_DIR
GRAPH_ROOT = DEFAULT_SD_5MIN_GRAPH_ROOT
OUTPUT_DIR = REPO_ROOT / 'benchmark' / 'eval' / 'sd_5min_delay_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 论文口径：先 spline 到更细粒度，再做 MCC
# 对 5min 数据，默认细化到 1min
INTERP_MINUTES = 1
MAX_LAG_MINUTES = 60

# 先跑一个月的整体分布
MONTH_START_DAY = 0
MONTH_NUM_DAYS = 31
MONTH_GRAPHS = ['distthre', 'phys_dir', 'phys_bidir']

# 再选其中一周，按天观察变化
WEEK_START_DAY = 0
WEEK_NUM_DAYS = 7
WEEK_GRAPHS = ['distthre', 'phys_dir', 'phys_bidir']


In [ ]:
flow, desc = load_dataset_flow(SD_5MIN_DIR)
sensor_ids = load_sensor_ids(SD_5MIN_DIR, int(desc['num_nodes']))
graphs = build_sd_5min_graph_matrices(SD_5MIN_DIR, GRAPH_ROOT)
native_minutes = int(desc['frequency (minutes)'])
native_minutes


In [ ]:
# Monthly slice
month_flow, month_meta = slice_flow_by_day_window(
    flow,
    native_minutes=native_minutes,
    start_day=MONTH_START_DAY,
    num_days=MONTH_NUM_DAYS,
)
month_meta


In [ ]:
delay_frames = []
for graph_name in MONTH_GRAPHS:
    delay_frames.append(
        compute_delay_distribution(
            graph_name=graph_name,
            adj=graphs[graph_name],
            flow=month_flow,
            sensor_ids=sensor_ids,
            native_minutes=native_minutes,
            interp_minutes=INTERP_MINUTES,
            max_lag_minutes=MAX_LAG_MINUTES,
            interpolation_method='natural_cubic_spline',
        )
    )

delay_df = pd.concat(delay_frames, ignore_index=True)
delay_df.head()


In [ ]:
delay_summary = summarize_delay_distribution(delay_df)
delay_summary


In [ ]:
delay_df.to_csv(OUTPUT_DIR / 'month_delay_distribution.csv', index=False)
delay_summary.to_csv(OUTPUT_DIR / 'month_delay_summary.csv', index=False)
print(OUTPUT_DIR / 'month_delay_distribution.csv')
print(OUTPUT_DIR / 'month_delay_summary.csv')


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(
    data=delay_df,
    x='delay_minutes',
    hue='graph',
    bins=20,
    element='step',
    common_norm=False,
)
plt.title('SD_5min Edge Delay Distribution (MCC-based)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'month_delay_hist.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=delay_df, x='graph', y='delay_minutes')
plt.title('SD_5min Edge Delay by Graph')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'month_delay_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(
    data=delay_df,
    x='corr_peak',
    hue='graph',
    bins=20,
    element='step',
    common_norm=False,
)
plt.title('SD_5min Peak Correlation Distribution of Delay Estimation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'month_corr_peak_hist.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Weekly daily breakdown
daily_frames = []
for graph_name in WEEK_GRAPHS:
    daily_frames.append(
        compute_daily_delay_distributions(
            graph_name=graph_name,
            adj=graphs[graph_name],
            flow=flow,
            sensor_ids=sensor_ids,
            native_minutes=native_minutes,
            start_day=WEEK_START_DAY,
            num_days=WEEK_NUM_DAYS,
            interp_minutes=INTERP_MINUTES,
            max_lag_minutes=MAX_LAG_MINUTES,
            interpolation_method='natural_cubic_spline',
        )
    )

daily_delay_df = pd.concat(daily_frames, ignore_index=True)
daily_delay_df.head()


In [ ]:
daily_summary = (
    daily_delay_df.groupby(['graph', 'day_offset'])
    .agg(
        num_edges=('delay_minutes', 'size'),
        delay_mean_min=('delay_minutes', 'mean'),
        delay_std_min=('delay_minutes', 'std'),
        delay_p50_min=('delay_minutes', lambda s: s.quantile(0.5)),
        delay_p90_min=('delay_minutes', lambda s: s.quantile(0.9)),
        corr_peak_mean=('corr_peak', 'mean'),
    )
    .reset_index()
)
daily_summary


In [ ]:
daily_delay_df.to_csv(OUTPUT_DIR / 'week_daily_delay_distribution.csv', index=False)
daily_summary.to_csv(OUTPUT_DIR / 'week_daily_delay_summary.csv', index=False)
print(OUTPUT_DIR / 'week_daily_delay_distribution.csv')
print(OUTPUT_DIR / 'week_daily_delay_summary.csv')


In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=daily_summary, x='day_offset', y='delay_mean_min', hue='graph', marker='o')
plt.title('Daily Mean Edge Delay Over Selected Week')
plt.xlabel('Day Offset From Dataset Start')
plt.ylabel('Mean Delay (min)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'week_daily_mean_delay.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=daily_delay_df, x='day_offset', y='delay_minutes', hue='graph')
plt.title('Daily Edge Delay Distribution Over Selected Week')
plt.xlabel('Day Offset From Dataset Start')
plt.ylabel('Delay (min)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'week_daily_delay_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()
